# Linear Regrssion:

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_percentage_error

## Importing Dataset:

In [2]:
data= pd.read_csv("Cleaned Laptop Dataset.csv")

In [3]:
data.drop(columns=['Unnamed: 0'], inplace=True)

## Building Base Model:

In [4]:
X = data.drop(columns=['Price'])
y = data['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [5]:
encode = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

X_train = encode.fit_transform(X_train)
X_test = encode.transform(X_test)

In [6]:
lr = LinearRegression(n_jobs=-1)

start_time = time.time()
lr.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.03s


## Results:

In [7]:
def adjusted_r2_score(y, y_pred):

    r2 = r2_score(y, y_pred)
    n = X.shape[0]
    k = X.shape[1]

    return 1 - ((1 - r2) * (n - 1) / (n - 1 - k))

In [8]:
# Results on Training Data:

y_pred_train = lr.predict(X_train)

print(f"r2 Score (Trainig Data): {r2_score(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nmean absolute error(Training Data): {mean_absolute_percentage_error(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_train, y_pred_train) * 100 :.2f}%")

r2 Score (Trainig Data): 79.64%

mean absolute error(Training Data): 21.98%

adjusted r2 Score: 79.58%


In [9]:
# Results on Testing Data:

y_pred_test = lr.predict(X_test)

print(f"r2 Score (Testing Data): {r2_score(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nmean absolute error (Testing Data): {mean_absolute_percentage_error(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_test, y_pred_test) * 100 :.2f}%")

r2 Score (Testing Data): 80.70%

mean absolute error (Testing Data): 23.03%

adjusted r2 Score: 80.65%


## Cross-Val-Score:

In [10]:
preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

lr_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', LinearRegression())
])

In [11]:
print(f"r2 Score: {np.mean(cross_val_score(estimator=lr_pipeline, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 78.42%


In [12]:
print(f"mean absolute error: {-np.mean(cross_val_score(estimator=lr_pipeline, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

mean absolute error: 22.81%


---

# Experimentaion:

## 1. Scaling (Input Features):

In [13]:
# Standard Scaler:

preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series']),
    ('scale', StandardScaler(), ['Weight', 'PPI', 'Clock Speed', 'RAM Speed', 'Refresh Rate', 'Graphics Memory', 'SSD', 'HDD', 'RAM', 'Missing RAM Speed', 'Missing Refresh Rate'])
], remainder='passthrough')

lr_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', LinearRegression())
])

print(f"r2 Score: {np.mean(cross_val_score(estimator=lr_pipeline, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

print(f"\nmean absolute error: {-np.mean(cross_val_score(estimator=lr_pipeline, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 78.42%

mean absolute error: 22.81%


## 2. Transformation (Input Features):

In [14]:
# Log Transformation:

preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series']),
    ('transform', FunctionTransformer(func=np.log1p), ['Weight', 'PPI', 'Clock Speed', 'RAM Speed', 'Refresh Rate', 'Graphics Memory', 'SSD', 'HDD', 'RAM', 'Missing RAM Speed', 'Missing Refresh Rate'])
], remainder='passthrough')

lr_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', LinearRegression())
])

print(f"r2 Score: {np.mean(cross_val_score(estimator=lr_pipeline, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

print(f"\nmean absolute error: {-np.mean(cross_val_score(estimator=lr_pipeline, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 74.66%

mean absolute error: 25.58%


In [15]:
# Yeo-Johnson Transformation:

preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series']),
    ('transform', PowerTransformer(method='yeo-johnson'), ['Weight', 'PPI', 'Clock Speed', 'RAM Speed', 'Refresh Rate', 'Graphics Memory', 'SSD', 'HDD', 'RAM', 'Missing RAM Speed', 'Missing Refresh Rate'])
], remainder='passthrough')

lr_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', LinearRegression())
])

print(f"r2 Score: {np.mean(cross_val_score(estimator=lr_pipeline, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

print(f"\nmean absolute error: {-np.mean(cross_val_score(estimator=lr_pipeline, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 74.25%

mean absolute error: 25.93%


## 3. Transformation (Target Feature):

In [16]:
preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

lr_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', LinearRegression())
])

lr_model = TransformedTargetRegressor(
    regressor=lr_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

print(f"r2 Score: {np.mean(cross_val_score(estimator=lr_model, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

print(f"\nmean absolute error: {-np.mean(cross_val_score(estimator=lr_model, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 72.13%

mean absolute error: 19.03%


## 4. Ridge Regularization:

In [17]:
preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

lr_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', Ridge())
])

param = {'model__alpha' : [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

grid_model = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

print(f"r2 Score: {np.mean(cross_val_score(estimator=grid_model, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

print(f"\nmean absolute error: {-np.mean(cross_val_score(estimator=grid_model, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 78.42%

mean absolute error: 22.84%


## 5. Lasso Regularization:

In [18]:
preprocessing = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

lr_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', Lasso())
])

param = {'model__alpha' : [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

grid_model = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

print(f"r2 Score: {np.mean(cross_val_score(estimator=grid_model, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

print(f"\nmean absolute error: {-np.mean(cross_val_score(estimator=grid_model, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 78.41%

mean absolute error: 22.73%


---

# Conclusion:

After experimenting with feature scaling, feature transformations, target transformation, and regularization, the **baseline Linear Regression model** was selected as the best overall configuration, achieving **{r2-Score = 78.42% & M.A.P.E = 22.81%}**.